# ANALYSE-09 : Un carreau de ℤ³ qui pave sans périodicité totale

**Série** : SymbolicAI / Lean — Digestions de résultats profonds, companion du capstone [Lean-20 — Digestions Tao](../Lean-20-Capstone-Digestions-Tao-Python.ipynb)
**Source** : famille 155 du corpus openai/math (« A counterexample to periodic tiling in dimension three », squelette Lean publié) — gisement de reconnaissance `G:\Mon Drive\MyIA\IA\Distillations\2026-10-openai-math\`
**Lac source** : `sudoku_lean` (la sémantique exact-cover / pavage discret du dépôt, `MyIA.AI.Notebooks/Sudoku/sudoku_lean/`)
**Kernel** : `python3` — le carnet illustre les définitions et mesure des cas jouets ; l'énoncé formel reste celui du corpus

## Navigation

| Notebook précédent | Notebook suivant |
|---|---|
| [ANALYSE-04 - PFR Primitives](ANALYSE-04-PFR-Primitives-Python.ipynb) | [Version anglaise](ANALYSE-09-Tuilage-Aperiodique_en.ipynb) |

***

## Pourquoi ce carnet

La conjecture du pavage périodique dit, grossièrement : *si une forme pave l'espace, elle le pave périodiquement*. Elle est vraie en dimension 1, vraie pour une tuile en dimension 2 (sur ℤ²), et **fausse à partir de la dimension 3** : la famille 155 du corpus openai/math exhibe un **carreau unique** — une partie finie T ⊂ ℤ³ — qui pave ℤ³ par translations mais dont **aucun pavage n'est totalement périodique**.

C'est le contre-exemple dans la **plus petite dimension possible** pour un carreau unique. L'épaississement en cubes unitaires transporte le même contre-exemple dans ℝ³, même en autorisant des vecteurs de translation réels arbitraires.

> **Statut épistémique** : openai/math est une source tierce non vérifiée par nous. L'énoncé ci-dessus est celui déclaré par le corpus (squelette Lean publié pour la famille 155) ; ce carnet le **présente** et l'**illustre** sur des cas jouets mesurés — il ne re-démontre pas le théorème.

## Ce que le carnet établit (plan)

1. **§1** — les définitions exactes : tuile, pavage par translations, période, *périodicité totale* (rang 3) — et pourquoi « apériodique » est plus subtil que « sans aucune période ».
2. **§2** — la connexion exact-cover : paver une fenêtre finie, c'est résoudre une couverture exacte — la sémantique que le lac `sudoku_lean` formalise.
3. **§3** — le moteur (backtracking borné + mesure du rang des périodes) et quatre expériences mesurées : un témoin périodique, une tuile qui ne pave **aucune** boîte (et pourquoi c'est un théorème), une tuile à période minimale **prouvée** égale à 4, et un piège : une tuile qui **semble** apériodique sur petite fenêtre.
4. **§4** — lecture honnête : ce que les fenêtres finies peuvent et ne peuvent pas établir, avec les mesures du §3 comme preuves.


## 1. Définitions — tuile, pavage, périodicité totale

**Tuile.** Une *tuile* est une partie **finie** non vide T ⊂ ℤ³. Ses éléments sont des cellules ; on pense à un polycube connexe, mais la définition n'exige pas la connexité.

**Pavage par translations.** La tuile T *pave* ℤ³ s'il existe un ensemble de translations S ⊂ ℤ³ tel que les copies T + s (s ∈ S) recouvrent ℤ³ **exactement** : chaque cellule de ℤ³ appartient à une et une seule copie. C'est une **couverture exacte** de ℤ³ par des copies translatées de T — ni trou, ni chevauchement.

**Période, pavage périodique.** Un vecteur v ∈ ℤ³ \ {0} est une *période* d'un pavage si l'ensemble de ses translations est invariant par v : translater tout le pavage de v le laisse inchangé. Le **groupe des périodes** d'un pavage est un sous-groupe de ℤ³ ; il a un **rang** r ∈ {0, 1, 2, 3} (dimension du ℤ-module).

**Périodicité totale.** Un pavage est *totalement périodique* quand r = 3 : trois périodes indépendantes. C'est la notion que le théorème 155 frappe : la tuile contre-exemple pave ℤ³, mais **aucun** de ses pavages n'atteint le rang 3.

**Pourquoi cette nuance compte.** « Apériodique » ne veut pas dire « zéro période » : un pavage peut admettre une ou deux directions de périodes (rang 1 ou 2) et rester un contre-exemple — ce qui est interdit, c'est le rang plein. Un pavage de rang 2, par exemple, se répète dans deux directions mais jamais dans la troisième : assez pour réfuter la conjecture, pas assez pour être « sans structure ».


## 2. La connexion exact-cover — ce que `sudoku_lean` formalise

Sur une **fenêtre finie** W (un bloc N₁×N₂×N₃ de ℤ³), paver W par des copies de T est un problème de couverture exacte : chaque cellule de W doit être couverte **exactement une fois**, chaque placement de tuile est un « choix », et les choix doivent être deux à deux non conflictuants.

C'est exactement la sémantique du lac `sudoku_lean` du dépôt : `ExactCover.IsExactCover` (`MyIA.AI.Notebooks/Sudoku/sudoku_lean/Sudoku/ExactCover.lean`) définit la couverture exacte d'un ensemble de *scopes* par une sélection, et le Sudoku y est l'instance canonique. Le Sudoku est un pavage : chaque case reçoit exactement un chiffre, chaque contrainte est couverte exactement une fois. Notre fenêtre de pavage est une autre instance de la même sémantique — cells × placements au lieu de cells × valeurs.

> **Copie pédagogique déclarée** (règle organ-first du dépôt) : le solveur borné de ce carnet est une réimplémentation locale **déclarée** — backtracking pur Python, lisible — de la sémantique exact-cover. L'organe natif (`sudoku_lean`) est un lac de preuves Lean, pas une librairie Python consommable ; l'invocation réelle n'est pas disponible côté Python, et le but ici est de montrer la mécanique, pas de la prouver. Réponses aux 5 questions : la série Sudoku possède la sémantique (1) ; son module n'est pas invocable depuis Python (2) ; l'exporter exigerait un port sans valeur pédagogique ici (3) ; le témoin négatif de l'organe est la preuve formelle `mem_toSelection_iff`, hors de portée d'un carnet (4) ; la vérification indépendante est le re-jeu multi-fenêtres du §3, dont les rangs mesurés confrontent les contraintes prouvées à la main (5).


## 3. Le moteur et les quatre expériences

Le moteur a deux organes, tous deux déterministes :

- `place_tiles(window, tile)` — backtracking exact-cover **borné** : la première cellule libre (ordre lexicographique) force le choix de l'ancre qui la couvre, on descend, on revient. Le compteur de nœuds mesure le coût combinatoire.
- `period_rank(placement, window)` — pour le pavage obtenu, teste tous les vecteurs de période v dans la boîte |vᵢ| ≤ ⌊Nᵢ/2⌋ (la **demi-fenêtre**) et retourne le rang sur ℤ du groupe qu'ils engendrent (élimination gaussienne exacte sur les rationnels).

La demi-fenêtre est la clef de lecture de tout le §3 : **toute période plus longue que la demi-fenêtre est invisible**. C'est un plafond de mesure, pas une propriété de la tuile — les expériences A et B2 le montrent chacun à leur façon.

Quatre expériences :

| # | Tuile | Ce qu'on mesure | Contrainte prouvable à la main |
|---|---|---|---|
| A | domino 1×1×2 | le témoin périodique : rang 3 attendu | pavage évident, périodes 1 ou 2 |
| B0 | prisme L×{0} | la tuile ne pave **aucune** boîte alignée | théorème : coin forcé, induction, case (1,1) morte |
| B1 | domino troué {(0,0,0),(2,0,0)} | rang 2 puis rang 3 quand la fenêtre grandit | période x minimale **exactement 4** dans tout pavage |
| B2 | cube dilaté {0,2}³ | rang 0 puis rang 3 | pavage de ℤ³ **unique**, périodes exactement 4ℤ³ |


In [1]:
"""Moteur de calcul pour le notebook ANALYSE-09-Tuilage-Aperiodique (famille 155).

Famille 155 : « contre-exemple au tuilage périodique en dimension trois » —
il existe une tuile translative finie T sous-ensemble de Z^3 qui tuile Z^3
mais n'admet AUCUN tuilage pleinement périodique (groupe de périodes de rang
3) ; des périodes partielles de rang < 3 peuvent exister. La tuile du théorème
est énorme : ce moteur fournit donc des mesures BORNÉES et HONNÊTES sur de
petites fenêtres.

Deux organes :
  * place_tiles  : exact-cover par backtracking borné (translations seules,
                   aucune rotation), déterministe, avec compteur de noeuds ;
  * period_rank  : rang sur Z du groupe des vecteurs de période du tuilage
                   RESTREINT à la fenêtre, testés dans |v_i| <= floor(N_i/2).

Limite pédagogique assumée (aussi dans tiling_measurements.json, champ
« limitation ») : une petite fenêtre ne peut jamais PROUVER l'apériodicité,
elle peut seulement la suggérer — une fenêtre plus grande peut révéler des
périodes jusqu'alors invisibles. La preuve du théorème vit dans la prose du
notebook, pas dans ces mesures.
"""

from __future__ import annotations

from fractions import Fraction
from itertools import product
from typing import Iterable, Optional

import numpy as np

Cell = tuple[int, int, int]
Placement = dict[Cell, Cell]  # cellule -> ancre de la copie qui la couvre



class _NodeBudgetExceeded(RuntimeError):
    """Budget de noeuds épuisé : instance abandonnée (ni échec ni succès)."""


def place_tiles(
    window_shape: tuple[int, int, int],
    tile_cells: Iterable[Cell],
    forbidden: Optional[set[Cell]] = None,
    *,
    max_nodes: int = 2_000_000,
) -> tuple[Optional[Placement], int]:
    """Tente de tuiler exactement la fenêtre par copies translatées de la tuile.

    Parameters
    ----------
    window_shape : (N1, N2, N3) — boîte [0,N1) x [0,N2) x [0,N3).
    tile_cells   : décalages (x,y,z) de la tuile, relatifs à son ancre.
    forbidden    : cellules à laisser vides (trous), optionnel.
    max_nodes    : garde-fou sur le nombre de noeuds de backtracking.

    Retour
    ------
    (placement, nodes) où placement vaut None si aucun tuilage n'est trouvé
    (ou si le budget de noeuds est épuisé). placement associe à chaque cellule
    couverte l'ancre de la copie qui la couvre — support direct du test de
    période de period_rank.

    Déterminisme : la première cellule libre est prise en ordre lexicographique
    (ordre C : x lent, z rapide) et les ancre candidates sont essayées par
    ordre croissant. Aucun aléatoire.
    """
    n1, n2, n3 = (int(v) for v in window_shape)
    if min(n1, n2, n3) < 1:
        raise ValueError("window_shape doit valoir >= 1 sur chaque axe")
    cells: list[Cell] = sorted({(int(x), int(y), int(z)) for x, y, z in tile_cells})
    if not cells:
        raise ValueError("tile_cells est vide")

    occ = np.zeros((n1, n2, n3), dtype=np.uint8)  # 0 = libre, 1 = couvert, 2 = trou
    if forbidden:
        for (x, y, z) in forbidden:
            occ[int(x), int(y), int(z)] = 2

    n_free = int((occ == 0).sum())
    if n_free % len(cells) != 0:
        return None, 1  # volume incompatible avec |T| : échec immédiat, décidé.

    anchors: list[Cell] = []
    nodes = 0

    def candidates(c: Cell) -> list[Cell]:
        """Ancres valides couvrant la cellule c, triées (déterminisme)."""
        out: list[Cell] = []
        for t in cells:
            a: Cell = (c[0] - t[0], c[1] - t[1], c[2] - t[2])
            if a in out:
                continue
            ok = True
            for u in cells:
                p = (a[0] + u[0], a[1] + u[1], a[2] + u[2])
                if not (0 <= p[0] < n1 and 0 <= p[1] < n2 and 0 <= p[2] < n3):
                    ok = False
                    break
                if occ[p] != 0:
                    ok = False
                    break
            if ok:
                out.append(a)
        return sorted(out)

    def solve() -> bool:
        nonlocal nodes
        nodes += 1
        if nodes > max_nodes:
            raise _NodeBudgetExceeded
        flat = occ.reshape(-1)
        k = int(np.argmin(flat))  # première cellule libre en ordre C
        if int(flat[k]) != 0:
            return True  # plus aucune cellule libre : succès
        c3 = np.unravel_index(k, (n1, n2, n3))
        c: Cell = (int(c3[0]), int(c3[1]), int(c3[2]))
        for a in candidates(c):
            placed = [(a[0] + u[0], a[1] + u[1], a[2] + u[2]) for u in cells]
            for p in placed:
                occ[p] = 1
            anchors.append(a)
            if solve():
                return True
            anchors.pop()
            for p in placed:
                occ[p] = 0
        return False

    try:
        solved = solve()
    except _NodeBudgetExceeded:
        return None, nodes
    if not solved:
        return None, nodes

    placement: Placement = {}
    for a in anchors:
        for u in cells:
            placement[(a[0] + u[0], a[1] + u[1], a[2] + u[2])] = a
    return placement, nodes


def period_rank(
    placement: Placement, window_shape: tuple[int, int, int]
) -> tuple[int, list[Cell]]:
    """Rang sur Z du groupe des périodes du tuilage, vues DANS la fenêtre.

    v est une période si, pour toute cellule c telle que c+v soit aussi dans
    la fenêtre, placement[c+v] == placement[c] + v (la copie qui couvre c,
    translatée de v, couvre c+v). Boîte de test bornée : |v_i| <= floor(N_i/2)
    (borne heuristique du sujet) ; les vecteurs v et -v sont identifiés et
    chaque période est mise sous forme canonique (premier coefficient non nul
    positif).

    Retour : (rang, generateurs) — sous-ensemble libre maximal pris dans
    l'ordre croissant (élimination gaussienne exacte sur Q via Fraction).
    """
    n1, n2, n3 = (int(v) for v in window_shape)
    anchor = np.zeros((n1, n2, n3, 3), dtype=np.int64)
    mask = np.zeros((n1, n2, n3), dtype=bool)  # cellules présentes dans le placement
    for c, a in placement.items():
        anchor[c] = a
        mask[c] = True

    dims = (n1, n2, n3)
    periods: list[Cell] = []
    ranges = product(*(range(-(n // 2), n // 2 + 1) for n in dims))
    for v in ranges:
        if v == (0, 0, 0):
            continue
        if any(n - abs(vi) < 1 for n, vi in zip(dims, v)):
            continue  # chevauchement vide : « période » vide de sens, on ignore
        src: list[slice] = []
        dst: list[slice] = []
        for n_i, v_i in zip(dims, v):
            if v_i >= 0:
                src.append(slice(0, n_i - v_i))
                dst.append(slice(v_i, n_i))
            else:
                src.append(slice(-v_i, n_i))
                dst.append(slice(0, n_i + v_i))
        shifted = anchor[tuple(dst)]
        base = anchor[tuple(src)] + np.array(v, dtype=np.int64)
        both = mask[tuple(dst)] & mask[tuple(src)]
        if not bool(np.any(both)):
            continue  # chevauchement réduit aux seuls trous : période vide de sens
        eq = np.all(shifted == base, axis=-1)  # égalité des ancres, axe vec inclus
        if bool(np.all(eq | ~both)):
            w = tuple(int(x) for x in v)
            for x in w:  # forme canonique : premier coefficient non nul positif
                if x < 0:
                    w = tuple(-y for y in w)
                    break
                if x > 0:
                    break
            if w not in periods:
                periods.append(w)

    # Préférence pour les générateurs courts (norme L1, puis ordre) : pure
    # lisibilité pédagogique, sans effet sur le rang ni le déterminisme.
    periods.sort(key=lambda w: (sum(abs(x) for x in w), w))
    gens = _independent_basis(periods)
    return len(gens), gens


def _independent_basis(vectors: list[Cell]) -> list[Cell]:
    """Sous-ensemble libre maximal, par élimination gaussienne exacte sur Q."""
    rows: list[list[Fraction]] = []
    pivots: list[int] = []
    kept: list[Cell] = []
    for v in vectors:
        row = [Fraction(x) for x in v]
        for piv, r in zip(pivots, rows):
            f = row[piv]
            if f:
                row = [a - f * b for a, b in zip(row, r)]
        for j, val in enumerate(row):
            if val:
                row = [a / val for a in row]
                rows.append(row)
                pivots.append(j)
                kept.append(v)
                break
    return kept


In [2]:
def mesurer(nom: str, cells: set[Cell], win: tuple[int, int, int]) -> dict[str, object]:
    """Resout une fenetre, mesure le rang des periodes, imprime et retourne."""
    placement, nodes = place_tiles(win, cells)
    entry: dict[str, object] = {
        "tile": nom, "cells": sorted(cells), "window": list(win),
        "nodes": nodes, "solved": placement is not None,
    }
    if placement is None:
        print(f" {nom} | fenetre {win} | nodes {nodes} | NON RESOLU")
        return entry
    rang, gens = period_rank(placement, win)
    entry["period_rank"] = rang
    entry["generators"] = [list(g) for g in gens]
    print(f" {nom} | fenetre {win} | nodes {nodes} | rang {rang} | generateurs {gens}")
    return entry


### 3.1 Expérience A — le domino, témoin périodique

Deux fenêtres : un cube 4×4×4 (le régime confortable) et une fenêtre volontairement **mince** (6,4,2) — la période (0,0,2) du domino y dépasse la demi-fenêtre z (= 1).


In [3]:
# A -- domino 1x1x2 : controle periodique.
domino: set[Cell] = {(0, 0, 0), (0, 0, 1)}
a_confort = mesurer("domino", domino, (4, 4, 4))
a_mince = mesurer("domino", domino, (6, 4, 2))


 domino | fenetre (4, 4, 4) | nodes 33 | rang 3 | generateurs [(0, 1, 0), (1, 0, 0), (0, 0, 2)]
 domino | fenetre (6, 4, 2) | nodes 25 | rang 2 | generateurs [(0, 1, 0), (1, 0, 0)]


### Lecture du résultat

Sur la fenêtre confortable (4,4,4), le groupe des périodes observé est de **rang 3** — trois générateurs indépendants dont (0,0,2) : le domino est totalement périodique, la mesure le voit. Sur la fenêtre mince (6,4,2), le rang mesuré tombe à **2** alors que la tuile n'a pas changé et reste périodique : la période (0,0,2) dépasse la demi-fenêtre z (⌊2/2⌋ = 1) et devient **invisible**. Le témoin périodique échoue donc lui-même à se montrer périodique quand la fenêtre est trop petite — c'est la première leçon de mesure du carnet : *le rang mesuré est un minorant du rang vrai, plafonné par la demi-fenêtre*.


### 3.2 Expérience B0 — le prisme en L : un échec de recherche qui est un théorème

Le triomino en L du plan xy ({(0,0,0), (1,0,0), (0,1,0)}), prismé avec une épaisseur 1 en z. On le lance sur trois fenêtres croissantes — il n'en résout **aucune**, en très peu de nœuds. Ce n'est pas un épuisement du solveur : c'est une **preuve**. Dans chaque plan z (la tuile étant plate), le coin (0,0) d'une boîte n'est couvrable que par l'ancre (0,0) — les deux autres candidats sortent — ce qui couvre (1,0) et (0,1) ; puis (0,2) n'est couvrable que par l'ancre (0,2), qui couvre (1,2) et (0,3) ; par induction toute la colonne 0 et les cases (1, pair) sont couvertes par des ancres forcées, et la case (1,1) n'a plus **aucun** candidat valide. Aucun rectangle 2D n'est tuilable par le L à orientation fixée, et le prisme plat hérite du résultat plan par plan.

Leçon pour la famille 155 : une tuile peut être **si** contraignante qu'elle interdit tout pavage — le contre-exemple 155 est l'équilibre inverse et subtil, une tuile qui pave ℤ³ mais interdit seulement la périodicité totale.


In [4]:
# B0 -- prisme L x {0} : ne tuile AUCUNE boite alignee (theoreme, cf. prose).
l_prisme: set[Cell] = {(0, 0, 0), (1, 0, 0), (0, 1, 0)}
b0 = [mesurer("L_prisme", l_prisme, w) for w in ((3, 3, 3), (4, 3, 3), (6, 4, 2))]


 L_prisme | fenetre (3, 3, 3) | nodes 4 | NON RESOLU
 L_prisme | fenetre (4, 3, 3) | nodes 4 | NON RESOLU
 L_prisme | fenetre (6, 4, 2) | nodes 5 | NON RESOLU


### 3.3 Expérience B1 — le domino troué : une période minimale prouvée, et l'artefact de fenêtre

Le domino troué {(0,0,0),(2,0,0)} : deux cellules séparées d'un trou. Sur chaque ligne x, chaque cellule est soit une ancre, soit une ancre + 2 (exactement une fois chacune) : notant f(n) ∈ {0,1} l'indicateur « n est une ancre », on a **f(n) + f(n−2) = 1 pour tout n**. La phase des ancres est donc exactement 4-périodique, et **aucune période x strictement inférieure à 4 n'existe dans aucun pavage** — une contrainte prouvée à la main, pas mesurée. Conséquence immédiate sur la mesure : une fenêtre de demi-largeur < 4 ne peut jamais certifier la période x. On mesure (4,2,2) puis (8,2,2).

### 3.4 Expérience B2 — le cube dilaté : le piège « semble apériodique »

Le cube dilaté {0,2}³ : les 8 coins d'un cube 2×2×2 espacé. Chaque copie ne couvre que des cellules de **sa classe de parité** (ajouter 2 préserve la parité de chaque coordonnée) ; par classe, le pavage se réduit (après division par 2) à un pavage par cubes unité. D'où un pavage de ℤ³ **unique** : ancres {0,1}³ + 4ℤ³, groupe de périodes **exactement 4ℤ³**. Sur (4,4,4), aucune période n'entre dans la boîte de test (|vᵢ| ≤ 2) : le rang mesuré tombe à **0** — l'allure « totalement apériodique » — alors que le vrai groupe est de rang 3. Sur (8,8,8), les trois générateurs (4,0,0), (0,4,0), (0,0,4) apparaissent.


In [5]:
# B1 -- domino troue : periode x minimale PROUVEE = 4 (cf. prose 3.3).
domino_troue: set[Cell] = {(0, 0, 0), (2, 0, 0)}
b1_etroite = mesurer("domino_troue", domino_troue, (4, 2, 2))
b1_large = mesurer("domino_troue", domino_troue, (8, 2, 2))

# B2 -- cube dilate : pavage de Z^3 unique, periodes exactement 4Z^3 (cf. prose 3.4).
cube_dilate: set[Cell] = {(x, y, z) for x in (0, 2) for y in (0, 2) for z in (0, 2)}
b2_petite = mesurer("cube_dilate", cube_dilate, (4, 4, 4))
b2_grande = mesurer("cube_dilate", cube_dilate, (8, 8, 8))


 domino_troue | fenetre (4, 2, 2) | nodes 9 | rang 2 | generateurs [(0, 0, 1), (0, 1, 0)]
 domino_troue | fenetre (8, 2, 2) | nodes 17 | rang 3 | generateurs [(0, 0, 1), (0, 1, 0), (4, 0, 0)]
 cube_dilate | fenetre (4, 4, 4) | nodes 9 | rang 0 | generateurs []
 cube_dilate | fenetre (8, 8, 8) | nodes 65 | rang 3 | generateurs [(0, 0, 4), (0, 4, 0), (4, 0, 0)]


### Lecture du résultat

Le domino troué : rang **2** sur (4,2,2) — les périodes partielles (0,1,0) et (0,0,1) seulement — puis rang **3** sur (8,2,2) où le générateur (4,0,0) devient visible, exactement la période minimale prouvée. Le cube dilaté pousse le même phénomène à l'extrême : rang **0** sur (4,4,4) — l'allure la plus apériodique qui soit — puis rang **3** sur (8,8,8) avec les trois générateurs de 4ℤ³. Les deux tuiles sont **périodiques** (leurs vrais groupes sont de rang 3) ; ce sont les fenêtres qui mentent, chacune dans l'autre sens :

- petite fenêtre ⇒ périodes invisibles ⇒ rang mesuré **trop petit** (B1, B2) ;
- la réciproque n'existe pas : un rang 3 mesuré est un **certificat** de périodicité totale (trois périodes vues), mais un rang < 3 n'est jamais un certificat d'apériodicité.


## 4. Ce qu'une fenêtre finie peut établir — et ce qu'elle ne peut pas

Une fenêtre finie ne peut **jamais prouver** l'apériodicité, pour une raison simple : tout pavage fini observé sur W est compatible avec un pavage de ℤ³ totalement périodique (répéter le motif de W). Les mesures du §3 le portent avec les chiffres :

- le **témoin** A lui-même rend rang 2 sur une fenêtre mince — le plafond de demi-fenêtre frappe même les tuiles les plus simples ;
- le domino troué (B1) et le cube dilaté (B2) ont des périodes minimales **prouvées** (4 en x pour l'un, 4ℤ³ exactement pour l'autre) : leurs rangs mesurés 2 et 0 sur petites fenêtres sont des **artefacts de visibilité**, levés quand la fenêtre double ;
- le prisme en L (B0) rappelle l'autre bord du spectre : une contrainte locale peut interdire **tout** pavage d'une boîte — l'échec du solveur y est une preuve, parce qu'elle est courte et structurée (ancres forcées), pas un épuisement.

Ce que la mesure apporte donc : **elle calibre** (le rang mesuré est un minorant plafonné par la demi-fenêtre), **elle compare** (le coût en nœuds du backtracking dit si la contrainte locale mord), **elle piège** (B2 montre qu'un rang 0 mesuré ne dit rien du vrai groupe). Ce qu'elle ne remplace pas : le théorème 155, qui établit l'absence de rang 3 pour **tous** les pavages de sa tuile — un quantificateur universel qu'aucune fenêtre finie n'atteint.

C'est la division du travail de la série ANALYSE : le carnet Python **mesure et illustre**, la formalisation **prouve** — ici, le squelette Lean publié par le corpus pour la famille 155.


## 5. Exercices

### Exercice 1 — le groupe des périodes, pas seulement son rang

La fonction `period_rank` rend le rang et des générateurs. Écrivez `period_group(placement, window_shape)` qui retourne la **liste complète** des vecteurs de période trouvés (sous forme canonique), puis vérifiez sur le domino que les périodes de norme L1 minimale sont (1,0,0), (0,1,0) et (0,0,2) — et expliquez pourquoi le pas est 2 et non 1 dans la direction longue.

```python
# TODO etudiant
def period_group(placement, window_shape):
    # Indice : reprendre la boucle de sondage de period_rank et collecter
    # TOUS les vecteurs v non nuls invariants, au lieu de s'arreter a une base.
    # Etape 1 : lister les candidats dans la demi-fenetre.
    # Etape 2 : tester l'invariance (comparaison vectorisee des ancres).
    # Etape 3 : mettre chaque periode sous forme canonique et dedupliquer.
    return None
```

### Exercice 2 — la plus petite fenêtre qui révèle le générateur x

Pour le domino troué, la période x minimale est 4 (prouvée au §3.3). Trouvez expérimentalement la plus petite fenêtre (N₁,2,2) à partir de laquelle le générateur (4,0,0) devient **visible** dans la mesure, et vérifiez que c'est bien N₁ = 8 (demi-fenêtre 4). Que se passe-t-il pour N₁ = 7 ?

```python
# TODO etudiant
def plus_petite_fenetre_revelante(tile_cells, axe=0):
    # Indice : balayer N1 croissant, appeler place_tiles puis period_rank,
    # s'arreter au premier rang 3. Attention au cas N1 impair (demi-fenetre
    # floor) : la reponse de N1=7 surprend-elle au vu de floor(7/2) = 3 ?
    return None
```

### Exercice 3 — construire une contrainte de parité

Le cube dilaté doit son pavage unique à un argument de **classe de parité**. Construisez une autre tuile dont le pavage de ℤ³ est unique par le même argument (par exemple un « domino dilaté » {(0,0,0),(2,0,0)} déjà vu — cherchez plutôt un triomino dilaté), prouvez l'unicité à la main, puis vérifiez que la mesure rend rang 0 sur petite fenêtre et rang 3 sur grande fenêtre, avec les générateurs attendus.

```python
# TODO etudiant
def triomino_dilate():
    # Indice : prendre les trois cellules du L, multiplier chaque coordonnee
    # par 2, et raisonner par classe de parite comme au 3.4.
    # Etape 1 : proposer la forme. Etape 2 : prouver l'unicite a la main.
    # Etape 3 : mesurer sur (4,4,4) puis (8,8,8) et confronter.
    return None
```


## Conclusion

### Ce qui a été établi

| Résultat | Mesure |
|---|---|
| Le domino est totalement périodique | rang 3 sur (4,4,4), générateurs dont (0,0,2) |
| Une fenêtre mince cache même les périodes évidentes | rang 2 sur (6,4,2) pour ce même domino |
| Le prisme en L ne pave aucune boîte alignée | 0/3 fenêtres résolues, échec court et structuré (théorème) |
| Le domino troué a une période x minimale prouvée de 4 | rang 2 sur (4,2,2) puis 3 sur (8,2,2), générateur (4,0,0) |
| Le cube dilaté a un pavage unique de groupe 4ℤ³ | rang 0 sur (4,4,4) puis 3 sur (8,8,8) |
| Une fenêtre finie ne prouve jamais l'apériodicité | argument du §4 : motif répété |

### Ce qui reste au théorème

L'énoncé 155 — un carreau fini de ℤ³ qui pave sans **aucun** pavage de rang 3 — n'est pas re-démontré ici : il est **présenté**, sourcé du corpus openai/math (squelette Lean publié), et illustré par des cas jouets mesurés dont chacun borne honnêtement sa portée. Le pont naturel est double : la formalisation côté Lean (le squelette du corpus), et l'exact-cover côté `sudoku_lean` — la sémantique est la même, seules les instances changent.

**Pistes pour la suite** : le passage ℝ³ (épaississement en cubes unitaires, translations réelles arbitraires) ; la comparaison avec les jeux de tuiles apériodiques classiques (Berger 1966, Jeandel–Rao en 2D — une *famille* de tuiles, là où 155 donne un carreau *unique*) ; et la complexité algorithmique du pavage (indécidabilité en général, le résultat de Berger).
